# Run Devang Model 2

Inference only. No training. This loads the fine-tuned LoRA adapter from Drive and runs the same Model 2 input/output schema used in `devangs_fine_tuned/examples`.

In [ ]:
from google.colab import drive

print('Mounting Google Drive...')
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
print('Installing dependencies...')
!pip -q install "torch>=2.0.0" "transformers>=4.40.0,<5" "peft>=0.13.0,<1" "accelerate>=1.0.0,<2" "bitsandbytes>=0.43" "pandas>=2.0" "numpy>=1.24"
print('Dependencies installed.')

In [ ]:
from pathlib import Path
import json
import os
import sys
import torch

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/devangs_fine_tuned')
ADAPTER_DIR = DRIVE_PROJECT_DIR / 'model2_v2_finetuned (1)'
EXAMPLE_INPUT = DRIVE_PROJECT_DIR / 'examples' / 'model2_input_example.json'
REFERENCE_OUTPUT = DRIVE_PROJECT_DIR / 'examples' / 'model2_output_example.json'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'outputs'
HF_CACHE_DIR = DRIVE_PROJECT_DIR / 'hf_cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR / 'transformers')

print('Project folder:', DRIVE_PROJECT_DIR)
print('Adapter folder:', ADAPTER_DIR)
print('Example input:', EXAMPLE_INPUT)
print('Output folder:', OUTPUT_DIR)
print('Hugging Face cache:', HF_CACHE_DIR)

assert DRIVE_PROJECT_DIR.exists(), f'Missing project folder: {DRIVE_PROJECT_DIR}'
assert ADAPTER_DIR.exists(), f'Missing adapter folder: {ADAPTER_DIR}'
assert (DRIVE_PROJECT_DIR / 'src').exists(), f'Missing src folder: {DRIVE_PROJECT_DIR / "src"}'
assert EXAMPLE_INPUT.exists(), f'Missing example input: {EXAMPLE_INPUT}'

sys.path.insert(0, str(DRIVE_PROJECT_DIR))

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: use a GPU runtime for this model.')

In [ ]:
required_adapter_files = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
]

print('Checking adapter files...')
missing = [name for name in required_adapter_files if not (ADAPTER_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing adapter files: {missing}')
print('Adapter files found.')

In [ ]:
from src.inference.engine import Model2Engine

print('Loading fine-tuned Model 2...')
print('Base model: Qwen/Qwen2.5-3B-Instruct')
print('Adapter:', ADAPTER_DIR)

engine = Model2Engine(
    adapter_path=str(ADAPTER_DIR),
    load_in_4bit=True,
)

print('Model loaded successfully.')

In [ ]:
print('Running folder example input...')

with open(EXAMPLE_INPUT, 'r', encoding='utf-8') as f:
    case_input = json.load(f)

result = engine.predict(case_input)

print('Inference complete.')
print('OK:', result['ok'])
print('Latency seconds:', round(result['latency_s'], 4))

if result['ok']:
    print(json.dumps(result['parsed'], indent=2, ensure_ascii=False))
else:
    print('Error:', result['error'])
    print('Raw text:', result['raw_text'])

In [ ]:
example_output_path = OUTPUT_DIR / 'example_prediction.json'

with open(example_output_path, 'w', encoding='utf-8') as f:
    json.dump({
        'ok': result['ok'],
        'prediction': result['parsed'],
        'raw_text': result['raw_text'],
        'error': result['error'],
        'latency_s': round(result['latency_s'], 4),
    }, f, indent=2, ensure_ascii=False)

print('Saved example prediction to:', example_output_path)

if REFERENCE_OUTPUT.exists():
    with open(REFERENCE_OUTPUT, 'r', encoding='utf-8') as f:
        reference_output = json.load(f)
    print('Reference output keys:', list(reference_output.keys()))

## Start API Server With Cloudflare Tunnel

Run the next cell after the model has loaded. It creates an API that accepts JSON directly, one uploaded JSON file, or multiple uploaded JSON files.

In [ ]:
print('Installing API dependencies...')
!pip -q install fastapi uvicorn python-multipart nest_asyncio

import re
import subprocess
import threading
import time
from typing import Any

import nest_asyncio
import uvicorn
from fastapi import Body, FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()

def strip_stable_id(record, id_column='case_id'):
    if isinstance(record, dict) and id_column in record:
        return {k: v for k, v in record.items() if k != id_column}, record[id_column]
    return record, None

def simple_summary(output):
    if not output['ok'] or not output['prediction']:
        return 'Prediction failed: ' + str(output['error'])
    p = output['prediction']
    return (
        f"Reason: {p.get('primary_reason')} | "
        f"Urgency: {p.get('urgency')} | "
        f"Action: {p.get('recommended_action')} | "
        f"Why: {p.get('reasoning_summary')}"
    )

def predict_one(record):
    case_payload, case_id = strip_stable_id(record)
    print('Running prediction...', 'case_id=' + str(case_id) if case_id else '')
    pred = engine.predict(case_payload)
    output = {
        'case_id': case_id,
        'ok': pred['ok'],
        'simple_output': None,
        'prediction': pred['parsed'],
        'raw_text': pred['raw_text'],
        'error': pred['error'],
        'latency_s': round(pred['latency_s'], 4),
    }
    output['simple_output'] = simple_summary(output)
    print(output['simple_output'])
    return output

def predict_payload(payload):
    if isinstance(payload, list):
        print('Received batch with', len(payload), 'records.')
        return [predict_one(record) for record in payload]
    if isinstance(payload, dict):
        print('Received single record.')
        return predict_one(payload)
    raise HTTPException(status_code=400, detail='Input must be a JSON object or a JSON list of objects.')

async def read_upload_json(file: UploadFile):
    content = await file.read()
    try:
        return json.loads(content.decode('utf-8'))
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f'Invalid JSON in {file.filename}: {exc}')

app = FastAPI(title='Devang Model 2 API')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/health')
def health():
    return {'ok': True, 'model_loaded': 'engine' in globals()}

@app.post('/predict-json')
def predict_json(payload: Any = Body(...)):
    return predict_payload(payload)

@app.post('/predict-file')
async def predict_file(file: UploadFile = File(...)):
    payload = await read_upload_json(file)
    return {'file_name': file.filename, 'result': predict_payload(payload)}

@app.post('/predict-files')
async def predict_files(files: list[UploadFile] = File(...)):
    results = []
    print('Received', len(files), 'files.')
    for file in files:
        payload = await read_upload_json(file)
        results.append({'file_name': file.filename, 'result': predict_payload(payload)})
    return results

print('Installing cloudflared...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

PORT = 8000
print(f'Starting API server on http://127.0.0.1:{PORT} ...')
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info'),
    daemon=True,
)
server_thread.start()
time.sleep(3)

print('Starting Cloudflare tunnel...')
cloudflared = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(80):
    line = cloudflared.stdout.readline().strip()
    if line:
        print(line)
        match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)
        if match:
            public_url = match.group(0)
            break

if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found. Rerun this cell.')

print('API server is ready.')
print('Public URL:', public_url)
print('Health:', public_url + '/health')
print('JSON endpoint:', public_url + '/predict-json')
print('Single file endpoint:', public_url + '/predict-file')
print('Multiple files endpoint:', public_url + '/predict-files')
print('Example curl for one file:')
print(f'curl -X POST -F "file=@model_2_input.json" {public_url}/predict-file')